# Pipeline Control Notebook
Use these interactive cells to run the Scraping, Transformation, and Ingestion stages manually.


In [7]:
import os
import sys
from pathlib import Path

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    from google.colab import auth
    auth.authenticate_user()  # Authenticate to GCP using your Google Login (No service account needed!)

    PROJECT_ROOT = '/content/drive/Othercomputers/Lenovo L15/Desktop/repo_collabs/ds_projects_collabs/2-nlp-astroturfing-report'
    if PROJECT_ROOT not in sys.path:
        sys.path.append(PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)

    # Provide the Project ID to the environment dynamically using Colab Secrets
    from google.colab import userdata
    try:
        os.environ['GCP_PROJECT_ID'] = userdata.get('GCP_PROJECT_ID')
    except Exception:
        os.environ['GCP_PROJECT_ID'] = input('Enter GCP Project ID: ')
else:
    # Running locally in VS Code
    current_dir = Path().absolute()
    project_root = current_dir.parent
    if str(project_root) not in sys.path:
        sys.path.append(str(project_root))

# Import modules
from src.forensics.manual_scrapping import scrape_submission_url
from src.infra.data_transformation import transform_to_structured
from src.infra.gcp_ingestion import load_to_bigquery


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Enter GCP Project ID: astroturfing-report


ModuleNotFoundError: No module named 'loguru'

# Phase 1. Scrapping and GCP upload.

## 1. Scrape a Reddit Thread
Replace the URL below with the submission you want to scrape.

In [ ]:
## WARNING this will take time, 1 sec per user scrapped from the URL used. ##
TARGET_URL = "https://www.reddit.com/r/mexico/comments/1rcb63u/i_hope_you_win_your_fight_against_the_cartel/" # Example URL

print(f"Starting manual scrape for: {TARGET_URL}")
raw_file_path = scrape_submission_url(TARGET_URL)

if raw_file_path:
    print(f"Scraping complete. Raw file saved at: {raw_file_path}")
else:
    print("Scraping failed.")


## 2. Transform Phase
Define your mode:
- `master`: Transforms all JSONs into a single master CSV (replaces BigQuery table entirely).
- `single`: Transforms only the latest raw file (appends to BigQuery table).

In [ ]:
MODE = "master"
print(f"Starting transformation in {MODE} mode...")

if MODE == "master":
    print("Consolidating all raw data into master CSV...")
    structured_file = transform_to_structured(
        input_path="data/raw",
        output_dir="data/structured",
        format="csv"
    )
    write_disposition = "WRITE_TRUNCATE"
else:
    print(f"Transforming single submission: {raw_file_path}")
    structured_file = transform_to_structured(
        input_path=raw_file_path,
        output_dir="data/structured",
        format="csv"
    )
    write_disposition = "WRITE_APPEND"

if structured_file:
    print(f"Transformation complete. Structured file ready at: {structured_file}")
else:
    print("Transformation failed.")


## 3. Ingestion Phase
Push the structured data to BigQuery.

In [ ]:
# Set the file to upload. To use an existing file instead of the pipeline output, edit these variables:
try:
    file_to_upload = structured_file
    current_write_disposition = write_disposition
except NameError:
    file_to_upload = r'C:\Users\arq_c\Desktop\repo_collabs\ds_projects_collabs\2-nlp-astroturfing-report\data\structured\transformed_comments.csv'
    current_write_disposition = 'WRITE_APPEND'

print(f"Syncing {file_to_upload} to BigQuery...")
print(f"Table operation mode: {current_write_disposition}")

try:
    load_to_bigquery(
        file_path=file_to_upload,
        table_name="comments_structured",
        write_disposition=current_write_disposition
    )
    print("Ingestion complete. Data is now in GCP.")
except Exception as e:
    print(f"Ingestion failed: {e}")


# Phase 2: Run GCP Transformations (Medallion Architecture)


## 1. BigQuery SQL implementations.
Executes the Bronze -> Silver -> Gold SQL pipelines in BigQuery.

In [ ]:
from src.infra.run_transformations import run_all_transformations

print("Starting GCP SQL Transformations...")
try:
    run_all_transformations()
    print("Transformations completed successfully!")
except Exception as e:
    print(f"Transformation failed: {e}")


2026-03-02 11:55:39.013 | INFO     | src.infra.run_transformations:run_sql_file:20 - Executing 01_bronze.sql...


Starting GCP SQL Transformations...


2026-03-02 11:55:41.220 | SUCCESS  | src.infra.run_transformations:run_sql_file:23 - Successfully executed 01_bronze.sql
2026-03-02 11:55:41.221 | INFO     | src.infra.run_transformations:run_sql_file:20 - Executing 02_silver.sql...
2026-03-02 11:55:43.616 | SUCCESS  | src.infra.run_transformations:run_sql_file:23 - Successfully executed 02_silver.sql
2026-03-02 11:55:43.617 | INFO     | src.infra.run_transformations:run_sql_file:20 - Executing 03_gold.sql...
2026-03-02 11:55:46.235 | SUCCESS  | src.infra.run_transformations:run_sql_file:23 - Successfully executed 03_gold.sql
2026-03-02 11:55:46.236 | INFO     | src.infra.run_transformations:run_sql_file:20 - Executing 04_gold_nlp.sql...
2026-03-02 11:55:48.375 | SUCCESS  | src.infra.run_transformations:run_sql_file:23 - Successfully executed 04_gold_nlp.sql


Transformations completed successfully!
